## Imports

In [61]:
import numpy as np

import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
import networkx as nx
import os
import pickle

from datetime import datetime
import time

import optuna


## LOADING THE DATA

In [62]:
path = '../data/final_dataset/graph/graphs_delaunay_norotation_local_new.pt'
#path = '../data/final_dataset/graph/graphs_sequential_norotation_local_new.pt'
dataset = torch.load(path, weights_only=False)
dataset = dataset[:20000]

In [63]:
# Split ratios
train_ratio = 0.5
val_ratio = 0.35 
test_ratio = 0.15
total = len(dataset)

batch_size = 4 #1,2,4,8

dataset = [data.sort(sort_by_row=False) for data in dataset]

train_dataset = dataset[:int(total * train_ratio)]
val_dataset   = dataset[int(total * train_ratio):int(total * (train_ratio + val_ratio))]
test_dataset  = dataset[int(total * (train_ratio + val_ratio)):]
print("Train Dataset Size: ", len(train_dataset))
print("Val Dataset Size: ", len(val_dataset))
print("Test Dataset Size: ", len(test_dataset))

if 'sequential' in path:
    sorted_indices = sorted(range(len(train_dataset)))
    train_dataset = [train_dataset[i] for i in sorted_indices]
    val_dataset   = [val_dataset[i] for i in sorted(range(len(val_dataset)))]
    test_dataset  = [test_dataset[i] for i in sorted(range(len(test_dataset)))]
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False) 
    print("Sequential dataset being used!")
else:
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) 
    print("Delaunay dataset being used!")

val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False) 
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Train Dataset Size:  10000
Val Dataset Size:  7000
Test Dataset Size:  3000
Delaunay dataset being used!


In [15]:
def plot_graph_with_targets_and_edges(data):
    num_nodes = data.num_nodes
    plt.figure(figsize=(8,7))

    # Extract base arrays
    coords = data.x[:, 1:3].detach().numpy()     # (num_nodes, 2)
    shifts = data.y[:, 0:2].detach().numpy()     # (num_nodes, 2)

    # Masks for node type (0 = original, 1 = synthetic_1)
    is_orig = (data.x[:, 0] == 0).detach().numpy()
    is_syn1 = (data.x[:, 0] == 1).detach().numpy()

    # Coordinates by type
    orig_coords = coords[is_orig]
    syn1_coords = coords[is_syn1]
    syn1_shifts = shifts[is_syn1]
    syn1_targets = syn1_coords + syn1_shifts  # synthetic_2 positions

    # --- Plot original polyline ---
    plt.plot(orig_coords[:, 0], orig_coords[:, 1],
             '-o', label='Original', markersize=2, color="#143642")

    # --- Plot synthetic_1 ---
    plt.plot(syn1_coords[:, 0], syn1_coords[:, 1],
             '-o', label='Synthetic 1 (input)', markersize=2, color="#EC9A29")

    # --- Plot synthetic_2 (targets) ---
    plt.plot(syn1_targets[:, 0], syn1_targets[:, 1],
             '-o', label='Synthetic 2 (target)', markersize=2, color="#A8201A")

    # --- Draw edges ---
    edge_index = data.edge_index.detach().numpy()
    for u, v in edge_index.T:   # each column is (src, dst)
        x1, y1 = coords[u]
        x2, y2 = coords[v]
        plt.plot([x1, x2], [y1, y2], linewidth=0.8, color="gray", alpha=0.4)

    # --- Draw arrows showing shifts ---
    for (x, y), (dx, dy) in zip(syn1_coords, syn1_shifts):
        plt.arrow(x, y, dx, dy,
                  head_width=0.002, head_length=0.004,
                  fc='gray', ec='gray', alpha=0.5)

    plt.title("Input + Target Polylines with Graph Edges")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.axis('equal')
    plt.legend()
    plt.show()


In [2]:
# for i in range(10,12):
#     plot_graph_with_targets_and_edges(train_dataset[i])

## TRAINING THE MODEL

In [64]:
# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
# MODEL ARCHITECTURE GraphSage with DELAUNAY

class GraphSAGEDelaunay(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.LayerNorm(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        #self.bn2 = nn.LayerNorm(out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        #x = self.bn2(x)
        return x

In [66]:
# MODEL ARCHITECTURE GraphSage with LSTM and SEQUENTIAL ORDERING

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        self.conv1 = SAGEConv(in_channels, hidden_channels, 'lstm')
        self.bn1 = nn.LayerNorm(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        #self.bn2 = nn.LayerNorm(out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        #x = self.bn2(x)
        return x

In [67]:
# MODEL PARAMETERS GraphSage 

sample = dataset[0]
in_channels = sample.num_features
out_channels = sample.y.shape[1]

# Model parameter
hidden_channels = 64
dropout = 0.4 #changed from 0.3
learning_rate = 1e-3
weight_decay = 1e-5

# Creating the model 
if 'delaunay' in path:
    model = GraphSAGEDelaunay(in_channels, hidden_channels, out_channels, dropout).to(device)
    description = 'delaunay_norotation_local_GSage_Huber_layernorm_newData' 
    print('Delaunay model selected!')

else:
    model = GraphSAGE(in_channels, hidden_channels, out_channels, dropout).to(device)
    description = 'sequential_norotation_local_GSage_Huber_lstm_layernorm_newData'
    print('Sequential model selected!')

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Loss Function
criterion = nn.SmoothL1Loss()
#criterion = nn.MSELoss()

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)


Delaunay model selected!


In [68]:
# EARLY STOPPING CALLBACK 

class EarlyStopping:
    def __init__(self, patience=20, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=15, delta=0.0001) #GraphSage 15 and delta 0.0001, 30 no delta

In [69]:
#SAVES MODEL STATE

def save_checkpoint(epoch, model, optimizer, train_losses, val_losses, path=''):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved to {path}")


In [70]:
#LOADS STATE TO RESUME TRAINING

def load_checkpoint(path, model, optimizer=None):
    checkpoint = torch.load(path, map_location="cpu")
    
    model.load_state_dict(checkpoint["model_state_dict"])
    
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    start_epoch = checkpoint["epoch"] + 1

    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    
    return start_epoch, train_losses, val_losses

In [71]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        
        mask = batch.x[:,0] == 1      
        loss = criterion(out[mask], batch.y[mask])
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        mask = batch.x[:,0] == 1      
        loss = criterion(out[mask], batch.y[mask])
        total_loss += loss.item()
    return total_loss / len(loader)


In [72]:
def train_and_evaluate_simple(
    hidden_channels,
    optimizer_name,
    learning_rate,
    batch_size,
    trial=None
):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=('delaunay' in path)
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Model
    if 'delaunay' in path:
        model = GraphSAGEDelaunay(
            in_channels, hidden_channels, out_channels, dropout=0.0
        ).to(device)
    else:
        model = GraphSAGE(
            in_channels, hidden_channels, out_channels, dropout=0.0
        ).to(device)

    # Optimizer
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    elif optimizer_name == "RMSprop":
        optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)
    else:
        raise ValueError(f"Unknown optimizer {optimizer_name}")

    criterion = nn.SmoothL1Loss()
    best_val_loss = float("inf")

    # Unique filename per trial (important!)
    model_path = (
        f"best_model_trial_{trial.number}.pt"
        if trial is not None
        else "best_model.pt"
    )

    for epoch in range(100):
        # ---- Training ----
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            mask = batch.x[:, 0] == 1
            loss = criterion(out[mask], batch.y[mask])
            loss.backward()
            optimizer.step()

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index)
                mask = batch.x[:, 0] == 1
                loss = criterion(out[mask], batch.y[mask])
                val_loss += loss.item()
        val_loss /= len(val_loader)

        # ---- Save best model ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)

    return best_val_loss

In [73]:
def objective(trial):
    hidden_channels = trial.suggest_categorical("hidden_channels", [32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "RMSprop"])
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [1, 2, 4, 8])

    val_loss = train_and_evaluate_simple(
        hidden_channels=hidden_channels,
        optimizer_name=optimizer_name,
        learning_rate=learning_rate,
        batch_size=batch_size
    )

    return val_loss

In [74]:
def print_status(study, trial):
    print(f"Trial {trial.number} finished.")
    print(f"  Validation Loss: {trial.value:.6f}")
    print(f"  Best Loss so far: {study.best_value:.6f}")
    print(f"  Best Hyperparameters so far: {study.best_params}\n")

In [75]:
study = optuna.create_study(
    direction="minimize",
    study_name="GraphSAGE_BO_Del"
)

study.optimize(
    objective, 
    n_trials=20,
    show_progress_bar=True, 
    callbacks=[print_status], 
    timeout=None
)

[I 2026-01-09 12:03:39,377] A new study created in memory with name: GraphSAGE_BO_Del
Best trial: 0. Best value: 0.00181213:   5%|▌         | 1/20 [18:24<5:49:52, 1104.88s/it]

[I 2026-01-09 12:22:04,256] Trial 0 finished with value: 0.0018121335530595388 and parameters: {'hidden_channels': 128, 'optimizer': 'RMSprop', 'lr': 0.0028456290516074146, 'batch_size': 4}. Best is trial 0 with value: 0.0018121335530595388.
Trial 0 finished.
  Validation Loss: 0.001812
  Best Loss so far: 0.001812
  Best Hyperparameters so far: {'hidden_channels': 128, 'optimizer': 'RMSprop', 'lr': 0.0028456290516074146, 'batch_size': 4}



Best trial: 0. Best value: 0.00181213:  10%|█         | 2/20 [34:19<5:04:57, 1016.55s/it]

[I 2026-01-09 12:37:58,970] Trial 1 finished with value: 0.002231701933504415 and parameters: {'hidden_channels': 32, 'optimizer': 'AdamW', 'lr': 6.959112238061083e-05, 'batch_size': 2}. Best is trial 0 with value: 0.0018121335530595388.
Trial 1 finished.
  Validation Loss: 0.002232
  Best Loss so far: 0.001812
  Best Hyperparameters so far: {'hidden_channels': 128, 'optimizer': 'RMSprop', 'lr': 0.0028456290516074146, 'batch_size': 4}



Best trial: 0. Best value: 0.00181213:  15%|█▌        | 3/20 [57:50<5:39:01, 1196.57s/it]

...

[I 2026-01-09 18:34:31,612] Trial 17 finished with value: 0.0018121025868999172 and parameters: {'hidden_channels': 128, 'optimizer': 'RMSprop', 'lr': 0.00023026024025768168, 'batch_size': 2}. Best is trial 12 with value: 0.0017508665280253938.
Trial 17 finished.
  Validation Loss: 0.001812
  Best Loss so far: 0.001751
  Best Hyperparameters so far: {'hidden_channels': 128, 'optimizer': 'RMSprop', 'lr': 0.00036918689111973364, 'batch_size': 1}



Best trial: 12. Best value: 0.00175087:  95%|█████████▌| 19/20 [6:48:17<21:05, 1265.51s/it]

[I 2026-01-09 18:51:56,516] Trial 18 finished with value: 0.001757026932892456 and parameters: {'hidden_channels': 128, 'optimizer': 'RMSprop', 'lr': 0.000514211797027233, 'batch_size': 4}. Best is trial 12 with value: 0.0017508665280253938.
Trial 18 finished.
  Validation Loss: 0.001757
  Best Loss so far: 0.001751
  Best Hyperparameters so far: {'hidden_channels': 128, 'optimizer': 'RMSprop', 'lr': 0.00036918689111973364, 'batch_size': 1}



Best trial: 12. Best value: 0.00175087: 100%|██████████| 20/20 [7:09:50<00:00, 1289.53s/it]

[I 2026-01-09 19:13:29,920] Trial 19 finished with value: 0.002006594109251767 and parameters: {'hidden_channels': 32, 'optimizer': 'RMSprop', 'lr': 0.0013511193912162569, 'batch_size': 1}. Best is trial 12 with value: 0.0017508665280253938.
Trial 19 finished.
  Validation Loss: 0.002007
  Best Loss so far: 0.001751
  Best Hyperparameters so far: {'hidden_channels': 128, 'optimizer': 'RMSprop', 'lr': 0.00036918689111973364, 'batch_size': 1}



In [76]:
print("Best validation loss:", study.best_value)
print("Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


Best validation loss: 0.0017508665280253938
Best hyperparameters:
  hidden_channels: 128
  optimizer: RMSprop
  lr: 0.00036918689111973364
  batch_size: 1


In [ ]:
best = study.best_params

final_val_loss = train_and_evaluate_simple(
    hidden_channels=best["hidden_channels"],
    optimizer_name=best["optimizer"],
    learning_rate=best["lr"],
    batch_size=best["batch_size"],
)

### TRAINING LOOP

In [ ]:
epochs = 300
train_losses, val_losses = [], []
best_val_loss = float("inf")
timestamp = datetime.now().strftime('%d%m_%H%M')
epochs_completed = 0
log_dir = f'../checkpoints/graph/logs/{timestamp}'
writer = SummaryWriter(log_dir)
epoch_times = []

# Reload and resume training
#start_epoch, train_losses, val_losses = load_checkpoint("checkpoint.pt", model, optimizer)

for epoch in range(1, epochs + 1):
    start = time.time()
    epochs_completed +=1
    train_loss = train_epoch(train_loader)
    val_loss = evaluate(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Log values to TensorBoard
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("LearningRate", scheduler.optimizer.param_groups[0]['lr'], epoch)

    scheduler.step(val_loss)        # LR scheduling
    early_stopping(val_loss)       # early stopping

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    # Save every epoch:
    #save_checkpoint(epoch, model, optimizer, train_losses, val_losses, "checkpoint.pt")

    # Or only save when validation improves:
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(epoch, model, optimizer, train_losses, val_losses, f'../checkpoints/graph/model/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.pt')

    epoch_times.append(time.time() - start)
    
    # if early_stopping.early_stop:
    #     print("Early stopping triggered.")
    #     break
    
#writer.add_graph(model, x_sample)
writer.close()
print("Avg time per epoch:", sum(epoch_times) / len(epoch_times))

In [ ]:
for name, param in model.named_parameters():
    writer.add_histogram(name, param, epoch)
    if param.grad is not None:
        writer.add_histogram(f'{name}.grad', param.grad, epoch)


In [ ]:
#%load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir ../checkpoints/graph/logs

In [ ]:
np.save(f'../checkpoints/graph/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs_train_loss', train_losses)
np.save(f'../checkpoints/graph/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs_val_loss', val_losses)

### PLOTTING TRAINING LOSS

In [ ]:
labels_font = {'family': 'serif', 'size': 10} #Times New Roman
tick_font = {'family': 'serif', 'size': 10}
title_font = {'family': 'serif', 'size': 14}
legend_font = {'family': 'serif', 'size': 10}

In [ ]:
# PLOT TRAINING & VALIDATION LOSS 

plt.figure(figsize=(8,6))
plt.plot(range(epochs_completed), train_losses[:epochs_completed], label='Train Loss', color='#143642') 
plt.plot(range(epochs_completed), val_losses[:epochs_completed], label='Validation Loss', color='#EC9A29')
plt.xlabel('Epoch', fontdict=labels_font)
plt.ylabel('Loss', fontdict=labels_font)
plt.title(f'Training & Validation Loss over {epochs_completed} Epochs ({description})', fontdict=title_font)
plt.legend(prop=legend_font)
plt.grid(True)

#save figure
save_path = f'../checkpoints/graph/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')

plt.show()

## POST TRAINING

In [ ]:
test_loss = evaluate(test_loader)
print("\nFinal Test Loss:", test_loss)

In [ ]:
# MAKE PREDICTIONS AND STORE THEM IN LIST
model.eval()

all_predictions = []  # list of dicts

with torch.no_grad():
    for data in test_dataset:
        data = data.to(device)

        pred_shift = model(data.x, data.edge_index).cpu().numpy()
        true_shift = data.y[:,0:2].cpu().numpy()
        coords = data.x[:,1:3].cpu().numpy()
        line_id = data.x[:,0].cpu().numpy()

        all_predictions.append({
            "coords": coords,
            "true_shift": true_shift,
            "pred_shift": pred_shift,
            "line_id": line_id
        })

In [ ]:
#SAVE RESULTS

with open(f'../data/results/graph/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.pkl', 'wb') as f:
    pickle.dump(all_predictions, f)


In [ ]:
def plot_predictions(pred_entry):
    coords = pred_entry["coords"]
    true_shift = pred_entry["true_shift"]
    pred_shift = pred_entry["pred_shift"]
    line_id = pred_entry["line_id"]

    orig_coords = coords[line_id == 0]
    syn1_coords = coords[line_id == 1]

    true_s2 = syn1_coords + true_shift[line_id == 1]
    pred_s2 = syn1_coords + pred_shift[line_id == 1]

    plt.figure(figsize=(8,7))

    # Original
    plt.plot(orig_coords[:,0], orig_coords[:,1],
             '-o', color="#143642", markersize=2, label="Original")

    # Synthetic-1
    plt.plot(syn1_coords[:,0], syn1_coords[:,1],
             '-o', color="#EC9A29", markersize=2, label="Synthetic 1 (input)")

    # Ground truth Synthetic-2
    plt.plot(true_s2[:,0], true_s2[:,1],
             '-o', color="#A8201A", markersize=2, label="Synthetic 2 (target)")

    # Predicted Synthetic-2
    plt.plot(pred_s2[:,0], pred_s2[:,1],
             '-o', color="#0F8B8D", markersize=2, label="Synthetic 2 (predicted)")

    # Shift arrows
    for i in range(len(syn1_coords)):
        plt.arrow(
            syn1_coords[i,0], syn1_coords[i,1],
            pred_s2[i,0] - syn1_coords[i,0],
            pred_s2[i,1] - syn1_coords[i,1],
            head_width=0.002, head_length=0.004,
            fc="#1C7C54", ec="#0F8B8D", alpha=0.4
        )

    plt.title("Prediction vs Ground Truth")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.axis("equal")
    plt.legend()
    plt.show()

In [ ]:
for i in range(500,501):
    sample_id = i
    plot_predictions(all_predictions[sample_id])
